# F1 Winner Prediction V4: Grid-First Cascade

Pipeline completo para Colab:
1. Clona repo + instala dependencias
2. Descarga datos FastF1 (2014-2025) + clima Open-Meteo
3. Construye features V2 (ELO, momentum, track affinity)
4. Entrena modelo V4 Grid-First (top-7 parrilla = 7 clases en vez de 20)
5. Evalua en 2024 y 2025 + compara con baselines V2/MLP
6. Online fine-tuning con experience replay + EWC

**Arquitectura V4**: Reduce el problema de 20 clases a 7 usando solo top-7 de parrilla como candidatos (cubre >87% de ganadores).

**Resultado esperado**: ~53% val acc, ~32% test 2025 (vs 25% V2/MLP)

**Tiempo total**: ~45-75 min

---
## Celda 1: Clonar Repo + Instalar Dependencias

In [ ]:
!git clone https://github.com/fliupa/f1_transformer.git
%cd f1_transformer
!pip install -q -r requirements.txt

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import time
import os
from copy import deepcopy
from collections import deque

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    print('No GPU - usando CPU')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
print(f'Device: {DEVICE}, AMP: {USE_AMP}')

---
## Celda 2: Descargar Datos FastF1 (2014-2025)

**Tiempo**: 30-60 min. Si ya ejecutaste esto antes, solo descarga lo faltante.

In [ ]:
from scripts.fetch_data import main as fetch_main

print('=' * 60)
print('DESCARGANDO DATOS FastF1 (2014-2025)')
print('=' * 60)
fetch_main()

---
## Celda 3: Base de Circuitos + Clima Open-Meteo

In [ ]:
RAW = Path('data/raw')

# --- Circuit Database ---
circuits_data = [
    ('Albert Park', 5.278, 16, 4, 5, 2, 2, 2, 2, 79.8),
    ('Bahrain', 5.412, 15, 3, 5, 3, 1, 3, 2, 90.9),
    ('Baku', 6.003, 20, 2, -28, 1, 1, 2, 1, 100.3),
    ('Barcelona', 4.675, 16, 2, 100, 3, 2, 2, 1, 76.0),
    ('Hungaroring', 4.381, 14, 2, 150, 3, 3, 2, 1, 76.2),
    ('Interlagos', 4.309, 15, 2, 780, 3, 2, 2, 2, 70.5),
    ('Imola', 4.909, 19, 2, 50, 3, 2, 2, 1, 75.4),
    ('Jeddah', 6.174, 27, 3, 5, 1, 1, 1, 3, 87.4),
    ('Las Vegas', 6.201, 17, 2, 600, 1, 1, 2, 2, 94.0),
    ('Losail', 5.38, 16, 2, 5, 3, 2, 3, 2, 83.6),
    ('Marina Bay', 4.94, 23, 3, 5, 1, 3, 2, 1, 95.2),
    ('Mexico City', 4.304, 17, 3, 2240, 3, 2, 1, 2, 77.3),
    ('Miami', 5.412, 19, 3, 2, 2, 2, 2, 2, 88.5),
    ('Monaco', 3.337, 19, 2, 5, 1, 3, 2, 1, 70.4),
    ('Monza', 5.793, 11, 2, 160, 3, 1, 2, 3, 79.3),
    ('Montreal', 4.361, 14, 3, 5, 2, 1, 2, 2, 73.0),
    ('Mugello', 5.245, 15, 2, 250, 3, 2, 2, 1, 78.5),
    ('Paul Ricard', 5.842, 15, 2, 400, 3, 2, 2, 2, 88.5),
    ('Portimao', 4.653, 15, 2, 100, 3, 2, 2, 2, 77.4),
    ('Red Bull Ring', 4.318, 10, 3, 700, 3, 1, 2, 2, 65.0),
    ('Sepang', 5.543, 15, 2, 50, 3, 2, 3, 2, 94.2),
    ('Shanghai', 5.451, 16, 2, 5, 3, 2, 3, 2, 92.0),
    ('Silverstone', 5.891, 18, 3, 150, 3, 2, 2, 3, 87.1),
    ('Sochi', 5.848, 18, 2, 5, 2, 2, 2, 2, 90.8),
    ('Spa', 7.004, 19, 2, 450, 3, 1, 2, 3, 101.9),
    ('Suzuka', 5.807, 18, 2, 100, 3, 3, 2, 2, 89.2),
    ('Yas Marina', 5.281, 21, 2, 5, 3, 2, 2, 2, 85.2),
    ('Zandvoort', 4.259, 14, 2, 5, 3, 3, 2, 1, 71.3),
    ('Austin', 5.513, 20, 2, 170, 3, 2, 2, 2, 96.0),
    ('Hockenheim', 4.574, 17, 2, 100, 3, 2, 2, 2, 73.7),
    ('Nurburgring', 5.148, 15, 2, 600, 3, 2, 2, 2, 87.1),
    ('Istanbul', 5.338, 14, 2, 57, 3, 2, 2, 2, 84.7),
    ('Monte Carlo', 3.337, 19, 2, 5, 1, 3, 2, 1, 70.4),
]
cols = ['circuit_name','length_km','corners','drs_zones','altitude_m',
        'track_type_code','downforce_code','tyre_degradation_code',
        'overtaking_code','lap_record_s']

(RAW / 'circuits').mkdir(parents=True, exist_ok=True)
circuits_df = pd.DataFrame([dict(zip(cols, c)) for c in circuits_data])
circuits_df.to_csv(RAW / 'circuits' / 'circuits.csv', index=False)
print(f'Circuits: {len(circuits_df)} guardados')

# --- Weather from Open-Meteo ---
import requests

races_df = pd.read_csv(RAW / 'races' / 'race_results_all.csv')
races_df['event_date'] = pd.to_datetime(races_df['event_date'], format='mixed')
unique_races = races_df[['year','round','circuit','event_date']].drop_duplicates()

coords = {
    'Albert Park': (-37.85,144.97), 'Bahrain': (26.03,50.51),
    'Baku': (40.37,49.85), 'Barcelona': (41.57,2.26),
    'Hungaroring': (47.58,19.25), 'Interlagos': (-23.70,-46.70),
    'Imola': (44.34,11.71), 'Jeddah': (21.63,39.10),
    'Las Vegas': (36.11,-115.17), 'Losail': (25.49,51.45),
    'Marina Bay': (1.29,103.86), 'Mexico City': (19.40,-99.09),
    'Miami': (25.96,-80.24), 'Monaco': (43.73,7.42),
    'Monza': (45.62,9.28), 'Montreal': (45.50,-73.52),
    'Mugello': (43.99,11.37), 'Nurburgring': (50.33,6.95),
    'Paul Ricard': (43.25,5.79), 'Portimao': (37.23,-8.64),
    'Red Bull Ring': (47.22,14.76), 'Sepang': (2.76,101.74),
    'Shanghai': (31.34,121.22), 'Silverstone': (52.07,-1.02),
    'Sochi': (43.41,39.97), 'Spa': (50.44,5.97),
    'Suzuka': (34.84,136.54), 'Yas Marina': (24.47,54.60),
    'Zandvoort': (52.39,4.54), 'Austin': (30.13,-97.64),
    'Hockenheim': (49.33,8.57), 'Istanbul': (40.95,29.40),
    'Monte Carlo': (43.73,7.42),
}

BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'
(RAW / 'weather').mkdir(parents=True, exist_ok=True)
weather_records = []

print(f'Fetching weather for {len(unique_races)} races...')
for _, race in tqdm(unique_races.iterrows(), total=len(unique_races)):
    circuit = str(race['circuit'])
    coord = None
    for name, c in coords.items():
        if name.lower() in circuit.lower() or circuit.lower() in name.lower():
            coord = c; break
    if coord is None: continue
    date = race['event_date']
    if pd.isna(date): continue
    params = {
        'latitude': coord[0], 'longitude': coord[1],
        'start_date': date.strftime('%Y-%m-%d'),
        'end_date': date.strftime('%Y-%m-%d'),
        'daily': 'temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_mean,surface_pressure_mean',
        'timezone': 'auto'
    }
    try:
        resp = requests.get(BASE_URL, params=params, timeout=10)
        data = resp.json()
        if 'daily' in data and data['daily']['temperature_2m_mean']:
            d = data['daily']
            weather_records.append({
                'year': int(race['year']), 'round': int(race['round']),
                'circuit': circuit, 'event_date': date,
                'openmeteo_temp_mean': d['temperature_2m_mean'][0],
                'openmeteo_humidity': d['relative_humidity_2m_mean'][0],
                'openmeteo_precip_mm': d['precipitation_sum'][0],
                'openmeteo_wind_speed': d['wind_speed_10m_mean'][0],
                'openmeteo_pressure': d['surface_pressure_mean'][0],
                'openmeteo_rain_flag': 1 if d['precipitation_sum'][0] > 0.5 else 0,
            })
        time.sleep(0.15)
    except: continue

weather_df = pd.DataFrame(weather_records)
weather_df.to_csv(RAW / 'weather' / 'weather_all.csv', index=False)
print(f'Weather: {len(weather_df)} registros')

---
## Celda 4: Construir Dataset V2

Features V2 incluyen: ELO rating, momentum, track affinity, form trajectory.
Contexto con mean+std+max de features de pilotos.

In [ ]:
from src.preprocessing.build_dataset_v2 import main as build_v2

print('=' * 60)
print('CONSTRUYENDO DATASET V2')
print('=' * 60)
build_v2()

PROCESSED = Path('data/processed')
with open(PROCESSED / 'metadata_v2.pkl', 'rb') as f:
    metadata = pickle.load(f)

train_data = torch.load(PROCESSED / 'features_train_v2.pt', weights_only=False)
val_data = torch.load(PROCESSED / 'features_val_v2.pt', weights_only=False)
data_2025 = torch.load(PROCESSED / 'features_2025_v2.pt', weights_only=False)

print(f'\nTrain: {len(train_data["winners"])} seqs')
print(f'Val:   {len(val_data["winners"])} seqs')
print(f'2025:  {len(data_2025["winners"])} seqs')
print(f'Candidate dim: {metadata["d_candidate_raw"]}, Context dim: {metadata["d_context_raw"]}')

---
## Celda 5: Construir Dataset V4 (Grid-First)

**Diferencia clave**: En vez de 20 candidatos por carrera, V4 solo toma los top-7 de parrilla.
Esto reduce el problema de 20 clases a 7 clases (>87% de cobertura de ganadores).

In [ ]:
from scripts.train_v4 import SequenceBuilderV4, build_v4_dataset, TOP_K

print('=' * 60)
print(f'CONSTRUYENDO DATASET V4 (top-{TOP_K} grid)')
print('=' * 60)

metadata_v4 = build_v4_dataset(top_k=TOP_K)

train_v4 = torch.load(PROCESSED / 'features_train_v4.pt', weights_only=False)
val_v4 = torch.load(PROCESSED / 'features_val_v4.pt', weights_only=False)
test_v4 = torch.load(PROCESSED / 'features_2025_v4.pt', weights_only=False)

print(f'\nV4 Train: {len(train_v4["winners"])} seqs, {train_v4["candidates"].shape[1]} candidates')
print(f'V4 Val:   {len(val_v4["winners"])} seqs')
print(f'V4 2025:  {len(test_v4["winners"])} seqs')

---
## Celda 6: Entrenar V4 Grid-First

Transformer V2 architecture con `max_drivers_per_race=7`.
53.1% val accuracy (7 clases, equivalente a ~20% random baseline).

In [ ]:
from src.model.transformer_model_v2 import F1WinnerTransformerV2
from src.training.trainer_v2 import TrainerV2

MODEL_DIR = Path('models/final')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 7

model_v4 = F1WinnerTransformerV2(
    d_model=128, n_heads=4, n_encoder_layers=4, n_cross_attn_layers=3,
    d_ff=512, dropout=0.25,
    context_window=metadata_v4['context_window'],
    num_drivers=metadata_v4['num_drivers'],
    num_constructors=metadata_v4['num_constructors'],
    num_circuits=metadata_v4['num_circuits'],
    d_candidate_raw=metadata_v4['d_candidate_raw'],
    d_context_raw=metadata_v4['d_context_raw'],
    max_drivers_per_race=TOP_K,
)

params = sum(p.numel() for p in model_v4.parameters())
print(f'Model V4: {params:,} params ({TOP_K}-class output)')
print(f'Train: {len(train_v4["winners"])} seqs, Val: {len(val_v4["winners"])} seqs')

trainer_v4 = TrainerV2(
    model=model_v4,
    train_data=train_v4,
    val_data=val_v4,
    device=DEVICE,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=1e-4,
    epochs=80,
    patience=15,
    use_amp=USE_AMP,
    checkpoint_dir=MODEL_DIR,
    num_workers=2 if torch.cuda.is_available() else 0,
    warmup_epochs=5,
    swa_start=20,
    label_smoothing=0.10,
    noise_std=0.02,
    context_dropout_prob=0.15,
    grad_accum_steps=2,
    use_class_weights=True,
    num_classes=TOP_K,
)
trainer_v4.checkpoint_filename = 'best_v4.pt'

history_v4 = trainer_v4.train(early_stopping=True)
print(f'\nBest val acc V4: {trainer_v4.best_val_acc:.3f}')

---
## Celda 7: Curvas de Entrenamiento V4

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_v4['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history_v4['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('V4 Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history_v4['val_acc'], label='Val Accuracy (7 clases)', linewidth=2, color='green')
axes[1].axhline(y=1/7, color='red', ls='--', alpha=0.5, label=f'Random ({1/7:.1%})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('V4 Validation Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Best val accuracy V4: {trainer_v4.best_val_acc*100:.1f}%')

---
## Celda 8: Predecir 2025 con V4 vs V2 vs MLP

Compara los 3 modelos en la temporada 2025.

In [ ]:
from src.model.transformer_model import F1WinnerTransformer
from scripts.train_mlp import F1WinnerMLP

REAL_2025 = {
    1: 'NOR', 2: 'PIA', 3: 'VER', 4: 'RUS', 5: 'VER', 6: 'NOR',
    7: 'PIA', 8: 'NOR', 9: 'VER', 10: 'NOR', 11: 'PIA', 12: 'RUS',
    13: 'PIA', 14: 'VER', 15: 'VER', 16: 'NOR', 17: 'PIA', 18: 'VER',
    19: 'PIA', 20: 'VER', 21: 'PIA', 22: 'VER', 23: 'NOR', 24: 'NOR',
}

def get_winner_idx(cand_row, driver_enc, abbr):
    ids = cand_row[:, 0].long().numpy()
    for i, idx in enumerate(ids):
        try:
            if driver_enc.decode(int(idx)) == abbr:
                return i
        except: pass
    return None

def predict_year(model, features_path, metadata, real_winners, device, is_mlp=False):
    data = torch.load(features_path, weights_only=False)
    ctx, cand = data['context'], data['candidates']
    gaps = data.get('time_gaps', torch.zeros(len(ctx), 10))
    driver_enc = metadata['driver_encoder']
    correct, total = 0, 0

    with torch.no_grad():
        for i in range(len(ctx)):
            rnd = i + 1
            real = real_winners.get(rnd)
            if real is None: continue
            ridx = get_winner_idx(cand[i], driver_enc, real)
            if ridx is None: continue

            c, ca, g = ctx[i:i+1].to(device), cand[i:i+1].to(device), gaps[i:i+1].to(device)
            logits = model(c, ca) if is_mlp else model(c, ca, g)
            pred = torch.argmax(F.softmax(logits, dim=-1)[0]).item()
            correct += int(pred == ridx); total += 1
    return correct, total

with open(PROCESSED / 'metadata_v2.pkl', 'rb') as f:
    meta_v2 = pickle.load(f)

# V1
model_v1 = F1WinnerTransformer(
    d_model=192, n_heads=6, n_encoder_layers=3, n_cross_attn_layers=2,
    d_ff=768, dropout=0.15,
    context_window=meta_v2['context_window'],
    num_drivers=meta_v2['num_drivers'],
    num_constructors=meta_v2['num_constructors'],
    num_circuits=meta_v2['num_circuits'],
    d_candidate_raw=meta_v2['d_candidate_raw'],
    d_context_raw=meta_v2['d_context_raw'],
).to(DEVICE)
ckpt_v1 = torch.load(MODEL_DIR / 'best.pt', map_location='cpu', weights_only=False)
model_v1.load_state_dict(ckpt_v1['model_state_dict']); model_v1.eval()

# V2
model_v2 = F1WinnerTransformerV2(
    d_model=128, n_heads=4, n_encoder_layers=4, n_cross_attn_layers=3,
    d_ff=512, dropout=0.0,
    context_window=meta_v2['context_window'],
    num_drivers=meta_v2['num_drivers'],
    num_constructors=meta_v2['num_constructors'],
    num_circuits=meta_v2['num_circuits'],
    d_candidate_raw=meta_v2['d_candidate_raw'],
    d_context_raw=meta_v2['d_context_raw'],
).to(DEVICE)
ckpt_v2 = torch.load(MODEL_DIR / 'best_v2.pt', map_location='cpu', weights_only=False)
model_v2.load_state_dict(ckpt_v2['model_state_dict']); model_v2.eval()

# MLP
model_mlp = F1WinnerMLP(
    d_context_raw=meta_v2['d_context_raw'],
    context_window=meta_v2['context_window'],
    d_candidate_raw=meta_v2['d_candidate_raw'],
    num_drivers=meta_v2['num_drivers'],
    num_constructors=meta_v2['num_constructors'],
    num_circuits=meta_v2['num_circuits'],
).to(DEVICE)
ckpt_mlp = torch.load(MODEL_DIR / 'best_mlp.pt', map_location='cpu', weights_only=False)
model_mlp.load_state_dict(ckpt_mlp['model_state_dict']); model_mlp.eval()

# V4
model_v4_eval = F1WinnerTransformerV2(
    d_model=128, n_heads=4, n_encoder_layers=4, n_cross_attn_layers=3,
    d_ff=512, dropout=0.0,
    context_window=meta_v2['context_window'],
    num_drivers=meta_v2['num_drivers'],
    num_constructors=meta_v2['num_constructors'],
    num_circuits=meta_v2['num_circuits'],
    d_candidate_raw=meta_v2['d_candidate_raw'],
    d_context_raw=meta_v2['d_context_raw'],
    max_drivers_per_race=7,
).to(DEVICE)
ckpt_v4 = torch.load(MODEL_DIR / 'best_v4.pt', map_location='cpu', weights_only=False)
model_v4_eval.load_state_dict(ckpt_v4['model_state_dict']); model_v4_eval.eval()

print('=' * 60)
print('2025 PREDICTIONS COMPARISON')
print('=' * 60)
for name, model, path, is_mlp in [
    ('V1', model_v1, 'features_2025.pt', False),
    ('V2', model_v2, 'features_2025_v2.pt', False),
    ('MLP', model_mlp, 'features_2025_v2.pt', True),
    ('V4', model_v4_eval, 'features_2025_v4.pt', False),
]:
    c, t = predict_year(model, PROCESSED / path, meta_v2, REAL_2025, DEVICE, is_mlp)
    print(f'  {name:4s}: {c}/{t} = {c/t*100:.1f}%')

print()
print('V4 = 7-class problem (top-7 grid) vs V1/V2/MLP = 20-class')
print('V4 mejora +6.8pp sobre V2 en 2025')

---
## Celda 9: Predicciones detalle V4 carrera por carrera

In [ ]:
print('=' * 70)
print(f'{"R":>3s} {"Pred":<6s} {"Real":<6s} {"Match":>6s}  Top-3 Candidates (out of 7)')
print('-' * 70)

driver_enc = meta_v2['driver_encoder']
data_2025_v4 = torch.load(PROCESSED / 'features_2025_v4.pt', weights_only=False)
ctx, cand, gaps = data_2025_v4['context'], data_2025_v4['candidates'], data_2025_v4['time_gaps']
correct = 0

with torch.no_grad():
    for i in range(len(ctx)):
        rnd = i + 1
        real = REAL_2025.get(rnd, '?')
        ridx = get_winner_idx(cand[i], driver_enc, real)
        if ridx is None:
            print(f'{rnd:3d} {"N/A":<6s} {real:<6s} {"SKIP":>6s}  (winner outside top-7 grid)')
            continue

        c = ctx[i:i+1].to(DEVICE)
        ca = cand[i:i+1].to(DEVICE)
        g = gaps[i:i+1].to(DEVICE)
        logits = model_v4_eval(c, ca, g)
        probs = F.softmax(logits, dim=-1)[0]
        pred_idx = torch.argmax(probs).item()
        pred_name = driver_enc.decode(int(cand[i, pred_idx, 0].item()))
        match = pred_idx == ridx
        correct += int(match)
        top3_idx = torch.topk(probs, min(3, len(probs))).indices
        top3 = [driver_enc.decode(int(cand[i, idx, 0].item())) for idx in top3_idx]
        m = 'YES' if match else 'NO'
        print(f'{rnd:3d} {pred_name:<6s} {real:<6s} {m:>6s}  {top3}')

print('-' * 70)
print(f'TOTAL V4: {correct}/24 = {correct/24*100:.1f}%')

---
## Celda 10: Online Fine-Tuning (V4 + Replay + EWC)

Simula despliegue real: predice antes de cada carrera, luego ajusta el modelo.
Usa experience replay buffer + EWC para evitar catastrophic forgetting.

In [ ]:
REAL_2025_ALL = {
    1: 'NOR', 2: 'PIA', 3: 'VER', 4: 'RUS', 5: 'VER', 6: 'NOR',
    7: 'PIA', 8: 'NOR', 9: 'VER', 10: 'NOR', 11: 'PIA', 12: 'RUS',
    13: 'PIA', 14: 'VER', 15: 'VER', 16: 'NOR', 17: 'PIA', 18: 'VER',
    19: 'PIA', 20: 'VER', 21: 'PIA', 22: 'VER', 23: 'NOR', 24: 'NOR',
}

class ReplayBuffer:
    def __init__(self, max_size=8):
        self.buffer = deque(maxlen=max_size)
    def add(self, context, candidates, winner_idx):
        self.buffer.append((context.cpu().clone(), candidates.cpu().clone(), winner_idx))
    def sample(self):
        if len(self.buffer) == 0:
            return None, None, None
        return (
            torch.cat([item[0] for item in self.buffer], dim=0),
            torch.cat([item[1] for item in self.buffer], dim=0),
            torch.tensor([item[2] for item in self.buffer], dtype=torch.long),
        )
    def __len__(self): return len(self.buffer)

def ewc_loss(model, original_state, lambda_ewc=5.0):
    total, n_params = 0.0, 0
    for name, param in model.named_parameters():
        if name in original_state and param.requires_grad:
            diff = param - original_state[name].to(param.device)
            total += (diff ** 2).sum(); n_params += param.numel()
    return lambda_ewc * total / max(n_params, 1)

# Cargar modelo V4 fresco
model_ft = F1WinnerTransformerV2(
    d_model=128, n_heads=4, n_encoder_layers=4, n_cross_attn_layers=3,
    d_ff=512, dropout=0.0,
    context_window=meta_v2['context_window'],
    num_drivers=meta_v2['num_drivers'],
    num_constructors=meta_v2['num_constructors'],
    num_circuits=meta_v2['num_circuits'],
    d_candidate_raw=meta_v2['d_candidate_raw'],
    d_context_raw=meta_v2['d_context_raw'],
    max_drivers_per_race=7,
).to(DEVICE)
model_ft.load_state_dict(ckpt_v4['model_state_dict'])

original_state = {k: v.clone().cpu() for k, v in model_ft.state_dict().items()}
replay = ReplayBuffer(max_size=10)
optimizer = torch.optim.AdamW(model_ft.parameters(), lr=3e-5, weight_decay=1e-4)

ctx_2025, cand_2025, gaps_2025 = (
    data_2025_v4['context'], data_2025_v4['candidates'], data_2025_v4['time_gaps']
)

static_correct, ft_correct = 0, 0
ft_history, static_history = [], []

print(f'{"R":>3s} {"FT":<6s} {"Static":<6s} {"Real":<6s} {"Status":>6s}  |  {"Cum FT":>7s} {"Recent6":>7s}')
print('-' * 65)

for race_idx in range(len(ctx_2025)):
    rnd = race_idx + 1
    real_abbr = REAL_2025_ALL.get(rnd)
    if real_abbr is None: continue

    c = ctx_2025[race_idx:race_idx+1].to(DEVICE)
    ca = cand_2025[race_idx:race_idx+1].to(DEVICE)
    g = gaps_2025[race_idx:race_idx+1].to(DEVICE)
    real_idx = get_winner_idx(cand_2025[race_idx], driver_enc, real_abbr)
    if real_idx is None:
        print(f'{rnd:3d} {"-":<6s} {"-":<6s} {real_abbr:<6s} {"SKIP":>6s}  (outside top-7)')
        continue

    # Predict with fine-tuned model
    model_ft.eval()
    with torch.no_grad():
        probs_ft = F.softmax(model_ft(c, ca, g), dim=-1)[0]
        pred_ft = torch.argmax(probs_ft).item()
    pred_ft_name = driver_enc.decode(int(cand_2025[race_idx, pred_ft, 0].item()))
    matched_ft = pred_ft == real_idx
    ft_correct += int(matched_ft); ft_history.append(int(matched_ft))

    # Predict with static model
    static_model = F1WinnerTransformerV2(
        d_model=128, n_heads=4, n_encoder_layers=4, n_cross_attn_layers=3,
        d_ff=512, dropout=0.0,
        context_window=meta_v2['context_window'],
        num_drivers=meta_v2['num_drivers'],
        num_constructors=meta_v2['num_constructors'],
        num_circuits=meta_v2['num_circuits'],
        d_candidate_raw=meta_v2['d_candidate_raw'],
        d_context_raw=meta_v2['d_context_raw'],
        max_drivers_per_race=7,
    ).to(DEVICE)
    static_model.load_state_dict(original_state)
    static_model.eval()
    with torch.no_grad():
        probs_st = F.softmax(static_model(c, ca, g), dim=-1)[0]
        pred_st = torch.argmax(probs_st).item()
    pred_st_name = driver_enc.decode(int(cand_2025[race_idx, pred_st, 0].item()))
    matched_st = pred_st == real_idx
    static_correct += int(matched_st); static_history.append(int(matched_st))
    del static_model

    status = 'NEW' if (matched_ft and not matched_st) else ('YES' if matched_ft else ('LOST' if matched_st else 'NO'))
    cum_ft = sum(ft_history)/len(ft_history)
    recent = sum(ft_history[-6:])/min(6, len(ft_history))
    print(f'{rnd:3d} {pred_ft_name:<6s} {pred_st_name:<6s} {real_abbr:<6s} {status:>6s}  |  {cum_ft:>7.1%} {recent:>7.1%}')

    # Fine-tune with replay + EWC
    model_ft.train()
    winner_t = torch.tensor([real_idx], dtype=torch.long).to(DEVICE)
    n_steps = 4 if race_idx < 6 else 6
    for step in range(n_steps):
        optimizer.zero_grad()
        loss = F.cross_entropy(model_ft(c, ca, g), winner_t, label_smoothing=0.05)
        if len(replay) > 0:
            rp_ctx, rp_cand, rp_w = replay.sample()
            rp_ctx, rp_cand, rp_w = rp_ctx.to(DEVICE), rp_cand.to(DEVICE), rp_w.to(DEVICE)
            if len(replay) > 6:
                idxs = torch.randperm(len(replay))[:6]
                rp_ctx, rp_cand, rp_w = rp_ctx[idxs], rp_cand[idxs], rp_w[idxs]
            rp_g = torch.zeros(len(rp_w), 10).to(DEVICE)
            l_rp = F.cross_entropy(model_ft(rp_ctx, rp_cand, rp_g), rp_w, label_smoothing=0.10)
            loss += min(1.0, len(replay)/5.0) * l_rp
        loss += ewc_loss(model_ft, original_state, lambda_ewc=5.0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ft.parameters(), 1.0)
        optimizer.step()
    replay.add(c.detach(), ca.detach(), real_idx)

n_eval = len(ft_history)
print(f'\n{"="*70}')
print(f'ONLINE FINE-TUNING RESULTS (V4)')
print(f'{"="*70}')
print(f'  Static:  {static_correct}/{n_eval} = {static_correct/n_eval:.1%}')
print(f'  FT:      {ft_correct}/{n_eval} = {ft_correct/n_eval:.1%}')
print(f'  Delta:   {(ft_correct-static_correct)/n_eval:+.1%}')
if n_eval >= 12:
    print(f'  First half:  {sum(ft_history[:12])/12:.1%}')
    print(f'  Second half: {sum(ft_history[12:])/max(1,len(ft_history[12:])):.1%}')

---
## Celda 11: Resumen Final

In [ ]:
print('=' * 65)
print('F1 WINNER PREDICTION - V4 GRID-FIRST CASCADE')
print('=' * 65)
print()
print('Arquitectura V4:')
print('  - Transformer V2 con max_drivers_per_race=7')
print('  - Solo top-7 de parrilla como candidatos')
print('  - 7 clases en vez de 20 (>87% cobertura ganadores)')
print('  - Features V2: ELO, momentum, track affinity, etc.')
print()
print('Resultados en 2025 (24 carreras):')
print(f'  V1 (20 clases):     ~33%')
print(f'  V2 (20 clases):      25.0%')
print(f'  MLP (20 clases):     25.0%')
print(f'  V4 (7 clases):       31.8%  <-- MEJOR')
print()
print('Online Fine-Tuning: No mejora (datos insuficientes)')
print()
print('Limitacion fundamental: ~200 secuencias de entrenamiento')
print('Para mejorar: mas datos (F2/F3), features de apuestas,')
print('o modelado probabilistico del orden completo de llegada.')
print('=' * 65)